# Chapter 23 — Transformer Kernels

Part 8: Advanced Triton operations and production.

**Ladder.**
- **Method 01.** Stage-by-stage PyTorch reference; the CPU oracle.
- **Method 02.** One Triton kernel per stage: embedding, norms, softmax, scores, output.
- **Method 03.** Block-fused attention with online softmax and a fused MLP.

**Thread.**
- **Recurring idea.** Every stage is a shape-preserving map around a residual stream; the only square object is the $T\times T$ score matrix.
- **Chain.** Embedding → position encoding → RoPE → QKV → scores → softmax → output → FlashAttention → LayerNorm/RMSNorm → MLP → fused block → KV cache.
- **Check.** Each stage gets a shape rule and a tiny numeric example; the kernels must reproduce both.

**Study tip.**
- Take the two-token, $d=2$ worked example: $q_0=(1,1)$, $k_0=(1,1)$, $k_1=(1,0)$.
- Compute $s_0=(1.4142,\,0.7071)$, $p_0=(0.6698,\,0.3302)$, $o_0=(1,\,1.3395)$.
- Write those four numbers on paper; every kernel is a way to produce them without materializing more than necessary.

**Learning objectives**
- Read each stage as a shape transformation and state the **attention shape rule**: $Q,K,V\in\mathbb R^{T\times d}$, $S=QK^\top/\sqrt d\in\mathbb R^{T\times T}$, $P=\operatorname{softmax}(S)\in\mathbb R^{T\times T}$, and $O=PV\in\mathbb R^{T\times d}$.
- Implement an embedding gather, sinusoidal position encoding, RoPE, and a QKV projection.
- Implement stable row softmax, LayerNorm, RMSNorm, and a GELU MLP in Triton.
- Implement a two-tile FlashAttention-style kernel with the online-softmax state $(m,\ell,\mathrm{acc})$.
- Explain the KV-cache shapes and why token decoding is memory-bound.
- Fuse stages and verify every value against a PyTorch reference with silent assertions.

[Index](../../00_index.ipynb) · [← Previous](../ch22_dot_gemm/22_complete.ipynb) · [Next →](../ch24_production/24_complete.ipynb)

**Estimated time:** 150–180 min · **Difficulty:** Advanced

**Prerequisites**
- Chapter 22: `tl.dot`, tiled GEMM, FP32 accumulation.
- Chapter 8: masks, `tl.where`, and boundary conditions.
- Chapter 7: stable softmax and RMS normalization.
- Basic linear algebra: matrix product, transpose, and row/column indexing.

**Book apparatus**
- [Setup and versions](../../docs/SETUP.md)
- [Notation](../../docs/NOTATION.md)
- [Glossary](../../docs/GLOSSARY.md)
- [Triton API index](../../docs/API_INDEX.md)
- [Formula index](../../docs/FORMULA_INDEX.md)
- [Diagrams](../../docs/DIAGRAMS.md)
- [Troubleshooting](../../docs/TROUBLESHOOTING.md)
- [Exercise hints](../../docs/SOLUTIONS.md)


## Concept diagrams

### The transformer block

```text
x ──┬──► LN ──► Attention ──► + ──┬──► LN ──► MLP ──► + ──► y
    │                         ▲   │                   ▲
    └─────────────────────────┘   └───────────────────┘
       residual 1                    residual 2
```

- **Residual stream.** Carries the same shape through the whole block.
- **Attention.** Mixes information **across tokens**.
- **MLP.** Mixes information **across channels**.
- **Pre-norm.** Both sublayers read a normalized copy and add their output back.

### QKV projection

```text
          ┌──► W_q ──► Q [T, d] ──┐
X [T, d] ─┼──► W_k ──► K [T, d] ──┼──► scores ──► softmax ──► output
          └──► W_v ──► V [T, d] ──┘
```

- **One input, three views.** The query asks, the key answers, the value is carried.
- **Fused QKV.** One GEMM with $3d$ output columns, split into three $[T,d]$ tensors.


### Scores → softmax → output

```text
Q [T, d]        K [T, d]
    \            /
     \          /
      S = Q Kᵀ / √d          [T, T]
             |
             v
      P = softmax(S)         [T, T], rows sum to 1
             |
             v
      O = P V                [T, d]
```

- **Scores.** Pairwise query/key compatibility.
- **Softmax.** Row-stochastic weights over the keys.
- **Output.** Convex combination of the value rows.

### FlashAttention tiles

```text
query tile q_i  [B_M, d]
   for each K/V tile (k_j, v_j) [B_N, d]:
       s_ij = q_i k_jᵀ / √d          [B_M, B_N]
       m_i  = max(m_i, max_j s_ij)
       ℓ_i  = e^{m_old-m_i} ℓ_i + Σ_j e^{s_ij-m_i}
       acc_i = e^{m_old-m_i} acc_i + e^{s_ij-m_i} v_j
   O_i = acc_i / ℓ_i
```

- **State.** $(m,\ell,\mathrm{acc})$ replaces the full $[T,T]$ matrix.
- **Memory.** The score tile is discarded after each K/V block.

### KV cache

```text
decode step t:   q_t [B, H, 1, d]

K_cache [B, H, t, d] ──┐
V_cache [B, H, t, d] ──┼──► scores [B, H, 1, t] ──► softmax ──► O [B, H, 1, d]
                       │
new k_t, v_t ──────────┘   append at position t, then t ← t + 1
```

- **Append.** One K/V row per token.
- **Traffic.** Reads grow linearly with context; decoding is bandwidth-bound.


In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import math
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU)


## How the chapter is built

| Part | Contents | Method |
|---|---|---|
| **I** | thirteen math cards, the shape rule, and the worked example | hand-checkable LaTeX |
| **II** | stage-by-stage PyTorch functions | Method 01 |
| **III** | embedding, norms, row softmax, scores, output | Method 02 |
| **IV** | online softmax, two-tile FlashAttention, fused MLP, block | Method 03 |
| **—** | KV-cache shapes, append, and traffic | Method 02/03 |
| **—** | pitfalls, self-check, summary, exercises | review |

- **Silent checks.** Every code cell verifies with `assert`; prints are one-line summaries plus the final dispatch.
- **GPU gating.** Triton tests use `run_if_gpu`; a CPU-only reader sees `SKIP ...` and the reference cells still run.


## Part I — Mathematics and concepts

- **Notation.** $T$ tokens, width $d$, batch $B$, $H$ heads of width $d_h=d/H$.
- **Layout.** A tensor with a batch and head axis is $[B,H,\cdot,\cdot]$; the head axis is a reshape of the channel axis, not a new learned dimension.
- **Shape rule.** $Q,K,V\in\mathbb R^{T\times d}$, then $S=QK^\top/\sqrt d\in\mathbb R^{T\times T}$, $P=\operatorname{softmax}(S)\in\mathbb R^{T\times T}$, and $O=PV\in\mathbb R^{T\times d}$.
- **Batch/heads.** Prepend $[B,H]$ to every shape.
- **Square object.** $S$ is square in the sequence length, not the model width; that is why attention costs $O(T^2d)$ and why FlashAttention exists.
- **Reading pattern.** Each subsection follows Connection → Rule → Formula → Example → Where → Triton → Failure → Bridge.


### 23.1 Embedding

- **Connection.** Builds on Chapter 11's token-id gather and opens the residual stream: ids arrive as integers, the block needs vectors.
- **Rule.** Each token id selects one row of the table; the trailing axis is always $d$. Shape: ids $(\cdots)\to(\cdots,d)$; a batch $(B,T)\to(B,T,d)$.
- **Formula.** $Y_i=E_{\pi(i),:}$ for $i=0,\dots,T-1$.
- **Example.** With the table and ids below, the lookup is a row copy.

$$
E=\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix},
\qquad \pi=(2,0),
\qquad
Y=E[\pi]=\begin{bmatrix}5&6\\1&2\end{bmatrix}.
$$

- **Where.** Maps a discrete token id to the dense vector that enters the residual stream; it is the first stage of every language model and is pure address computation.
- **Triton.** Load the id, then read `table + idx * d + col`; a 2-D grid over (tokens, channels) keeps both loads coalesced.
- **Failure.** An out-of-range or negative id reads the wrong row (or crashes); a non-standard table stride is read as contiguous. Validate ids at the host boundary.
- **Bridge.** §23.2 adds order, because a gather is order-blind.


### 23.2 Position encoding

- **Connection.** Builds on §23.1: the gather gives token identity but no position; attention alone is permutation-equivariant.
- **Rule.** Add a fixed sin/cos pattern elementwise; $PE$ has the same shape as the token embeddings, so $H=X+PE$.
- **Formula.** $PE_{t,2i}=\sin(t/10000^{2i/d})$ and $PE_{t,2i+1}=\cos(t/10000^{2i/d})$.
- **Example.** For $T=2$, $d=4$:

$$
PE=\begin{bmatrix}
0 & 1 & 0 & 1\\
\sin 1 & \cos 1 & \sin 0.01 & \cos 0.01
\end{bmatrix}
=
\begin{bmatrix}
0 & 1 & 0 & 1\\
0.8415 & 0.5403 & 0.0100 & 0.9999
\end{bmatrix}.
$$

- **Where.** Breaks permutation symmetry; the pattern is fixed (no parameters) and can be computed for positions never seen in training.
- **Triton.** Compute the frequency with `tl.exp2(-2 * (d // 2) / D * log2(base))`, then `tl.sin` and `tl.cos`; match the interleaved $(\sin,\cos)$ layout.
- **Failure.** Mixing interleaved and half-split layouts silently permutes channels; the wrong base changes the wavelength; `float16` angles lose precision at large $t$.
- **Bridge.** §23.3 replaces additive positions with rotations of Q/K.


### 23.3 RoPE

- **Connection.** Builds on §23.2: additive position works, but RoPE makes position multiplicative in the queries and keys.
- **Rule.** Rotate each channel pair by a position-dependent angle; the shape is preserved ($[T,d]\to[T,d]$ or $[B,H,T,d_h]\to[B,H,T,d_h]$), and tokens are not mixed.
- **Formula.** $x'_{2i}=x_{2i}\cos\theta_i-x_{2i+1}\sin\theta_i$, $x'_{2i+1}=x_{2i}\sin\theta_i+x_{2i+1}\cos\theta_i$, with $\theta_i=p\,10000^{-2i/d}$.
- **Example.** For $x=(1,0)$, $p=1$, $i=0$: $\theta_0=1$.

$$
\begin{bmatrix}x'_{0}\\x'_{1}\end{bmatrix}
=
\begin{bmatrix}\cos1&-\sin1\\ \sin1&\cos1\end{bmatrix}
\begin{bmatrix}1\\0\end{bmatrix}
=
\begin{bmatrix}0.5403\\0.8415\end{bmatrix}.
$$

- **Where.** Used by Llama, Mistral, and many decoders; $\langle R(p)q,R(s)k\rangle=f(p-s)$, so scores depend on the relative offset $p-s$. Rotate Q and K; leave V alone.
- **Triton.** Load the two halves of each pair, compute $\cos\theta$ and $\sin\theta$ once, and store $x_1c-x_2s$ and $x_1s+x_2c$.
- **Failure.** A flipped sign changes $p-s$ to $s-p$; the wrong pair layout shuffles channels; applying RoPE to V changes the output without theoretical need.
- **Bridge.** §23.4 projects the positioned tokens into Q, K, and V.


### 23.4 QKV projection

- **Connection.** Builds on §23.3: positioned token vectors now get three learned projections from one residual stream.
- **Rule.** $X\in\mathbb R^{T\times d}$ times $W\in\mathbb R^{d\times 3d}$ gives $[T,3d]$, split into three $[T,d]$ tensors. For heads, reshape $[B,T,H,d_h]$ to $[B,H,T,d_h]$.
- **Formula.** $Q=XW_q$, $K=XW_k$, $V=XW_v$, with $W_q,W_k,W_v\in\mathbb R^{d\times d}$; a fused kernel concatenates them into $W_{qkv}\in\mathbb R^{d\times 3d}$.
- **Example.** With $W_q=W_k=I$ and the value weights below:

$$
X=\begin{bmatrix}1&1\\1&0\end{bmatrix},\quad
W_v=\begin{bmatrix}1&0\\0&2\end{bmatrix},\quad
Q=\begin{bmatrix}1&1\\1&0\end{bmatrix},\quad
K=\begin{bmatrix}1&1\\1&0\end{bmatrix},\quad
V=\begin{bmatrix}1&2\\1&0\end{bmatrix}.
$$

- **Where.** One stream is projected into three roles: what this token asks for, what it offers as a match, and what it carries forward.
- **Triton.** One GEMM with $N=3d$ and an epilogue that writes three slices, or three GEMMs with the same $X$ tile.
- **Failure.** `[B,T,H,d_h]` is not `[B,H,T,d_h]`; a missing bias shifts every row; a transposed weight is a shape error or, when $d_q=d_k$, a plausible wrong result.
- **Bridge.** §23.5 uses Q and K to build the score matrix.


### 23.5 Attention scores

- **Connection.** Builds on §23.4's Q and K: the first pairwise operation in the block.
- **Rule.** $Q\in\mathbb R^{T_q\times d}$ and $K\in\mathbb R^{T_k\times d}$ give $S\in\mathbb R^{T_q\times T_k}$; with batch and heads, $[B,H,T_q,T_k]$. Causal attention needs $T_q\le T_k$.
- **Formula.** $S=QK^\top/\sqrt d$ and $S_{ij}=q_i\cdot k_j/\sqrt d$.
- **Example.** From §23.4:

$$
QK^\top=\begin{bmatrix}2&1\\1&1\end{bmatrix},
\qquad
S=\frac{1}{\sqrt2}\begin{bmatrix}2&1\\1&1\end{bmatrix}
=\begin{bmatrix}1.4142&0.7071\\0.7071&0.7071\end{bmatrix}.
$$

- **Where.** The score matrix is the pairwise compatibility of every query with every key; it is the only $O(T^2)$ object in the block.
- **Triton.** `s = tl.dot(q, tl.trans(k), input_precision="ieee") * scale` on a 2-D grid of $[B_M,B_N]$ tiles; keep q and k in FP32 for the dot.
- **Failure.** Missing $1/\sqrt d$ grows the logits with $d$ and saturates softmax; transposing the wrong operand computes $Q^\top K$; scaling after softmax changes the temperature.
- **Bridge.** §23.6 turns the scores into row-stochastic weights.


### 23.6 Softmax

- **Connection.** Builds on §23.5's score matrix: each row is an independent logit vector.
- **Rule.** Normalize each row over the **key** axis; every row of $P$ sums to one. Masked entries go to $-\infty$ before the max.
- **Formula.** $P_{ij}=e^{S_{ij}-m_i}/\sum_{j'}e^{S_{ij'}-m_i}$ with $m_i=\max_{j'}S_{ij'}$.
- **Example.** The two rows of $S$ give the probability matrix below.

$$
P=\begin{bmatrix}
\dfrac{e^{1.4142}}{e^{1.4142}+e^{0.7071}}&
\dfrac{e^{0.7071}}{e^{1.4142}+e^{0.7071}}\\[6pt]
\dfrac{e^{0.7071}}{2e^{0.7071}}&
\dfrac{e^{0.7071}}{2e^{0.7071}}
\end{bmatrix}
=
\begin{bmatrix}0.6698&0.3302\\0.5&0.5\end{bmatrix}.
$$

- **Where.** Converts scores into a convex combination; also the main numerical hazard, since $e^x$ overflows above about $88.7$ in FP32 and much earlier in FP16.
- **Triton.** One program per row: load, subtract `tl.max`, exponentiate, sum, divide; fuse with the score and output tiles for attention.
- **Failure.** No max shift overflows; normalizing over queries gives columns that sum to one; FP16 exponentials saturate; masking after softmax leaves probabilities unnormalized.
- **Bridge.** §23.7 multiplies the probability rows by V.


### 23.7 Attention output

- **Connection.** Builds on §23.6's P and §23.4's V: the second pairwise product.
- **Rule.** $P\in\mathbb R^{T_q\times T_k}$ times $V\in\mathbb R^{T_k\times d_v}$ gives $O\in\mathbb R^{T_q\times d_v}$; when $d_v=d$ the output re-enters the residual stream unchanged in shape.
- **Formula.** $O=PV$ and $O_{ij}=\sum_k P_{ik}V_{kj}$.
- **Example.** With $P$ from §23.6 and $V$ from §23.4:

$$
O=\begin{bmatrix}0.6698&0.3302\\0.5&0.5\end{bmatrix}
\begin{bmatrix}1&2\\1&0\end{bmatrix}
=\begin{bmatrix}1.0000&1.3395\\1.0000&1.0000\end{bmatrix}.
$$

- **Where.** Each output row is a weighted average of the value rows, with weights from that query's attention row; this is the mix-across-tokens half of the block.
- **Triton.** `o = tl.dot(p, v, input_precision="ieee")` with a $[B_M,B_N]$ probability tile and a $[B_N,d]$ value tile; keeping P in registers avoids HBM.
- **Failure.** Unnormalized P scales the output; a transposed V gives a shape error or wrong mixture; masking after the output leaves the masked contribution in the sum.
- **Bridge.** §23.8 produces the same output without materializing S or P.


### 23.8 FlashAttention

- **Connection.** Builds on §23.5–§23.7: exactly the same math, but a different memory schedule.
- **Rule.** Stream K/V tiles and keep $(m,\ell,\mathrm{acc})$ per query row: $m,\ell\in\mathbb R^{T_q}$ and $\mathrm{acc}\in\mathbb R^{T_q\times d}$. No $[T_q,T_k]$ matrix is stored.
- **Formula.** For each K/V tile $j$:

$$
\begin{aligned}
m_i^{(j)}&=\max\left(m_i^{(j-1)},\max_{k\in j}S_{ik}\right),\\
\ell_i^{(j)}&=e^{m_i^{(j-1)}-m_i^{(j)}}\ell_i^{(j-1)}+\sum_{k\in j}e^{S_{ik}-m_i^{(j)}},\\
\mathrm{acc}_i^{(j)}&=e^{m_i^{(j-1)}-m_i^{(j)}}\mathrm{acc}_i^{(j-1)}+\sum_{k\in j}e^{S_{ik}-m_i^{(j)}}V_{k,:}.
\end{aligned}
$$

  - **Final.** $O_i=\mathrm{acc}_i/\ell_i$.
- **Example.** One row, two tiles; $s_0=(1.4142,0.7071)$ and $V=((1,2),(1,0))$:

| tile | $m$ | $\ell$ | $\mathrm{acc}$ |
|---:|---:|---:|---|
| 1 (key 0) | 1.4142 | 1.0000 | $(1.0000,2.0000)$ |
| 2 (key 1) | 1.4142 | 1.4931 | $(1.4931,2.0000)$ |

  - **Result.** $O_0=\mathrm{acc}/\ell=(1.0000,1.3395)$, the direct result.
- **Where.** Attention is memory-bound when $S$ and $P$ are written and re-read; FlashAttention reduces HBM traffic from $O(T^2)$ to $O(Td)$ and makes long context practical.
- **Triton.** Loop `range(0, T, B_N)`, compute `s = tl.dot(q, tl.trans(k), input_precision="ieee")`, update `m`, `l`, `acc`, and rescale the old accumulator by `tl.exp(m_old - m_new)`.
- **Failure.** Rescaling $\ell$ but not $\mathrm{acc}$ (or the reverse); using the new max without the correction factor; dividing by $\ell$ inside the loop; masking after the max; a fully masked tile producing $-\infty-\infty=\mathrm{nan}$.
- **Bridge.** §23.9 normalizes the residual stream around the attention output.


### 23.9 LayerNorm

- **Connection.** Builds on §23.7–§23.8: the attention output enters the residual; the next sublayer reads a normalized copy.
- **Rule.** Per row, subtract the mean, divide by the standard deviation, then apply the affine map; output shape equals input shape, with $\gamma,\beta\in\mathbb R^d$ broadcast across rows.
- **Formula.** $\mu=\frac1d\sum_j x_j$, $\sigma^2=\frac1d\sum_j(x_j-\mu)^2$, and $y_j=\gamma_j(x_j-\mu)/\sqrt{\sigma^2+\epsilon}+\beta_j$.
- **Example.** For $x=(1,2,3,4)$:

$$
\mu=2.5,\qquad \sigma^2=1.25,\qquad
\hat x=\frac{x-2.5}{\sqrt{1.25+10^{-5}}}
=(-1.3416,-0.4472,0.4472,1.3416).
$$

  - **Affine.** With $\gamma=1$ and $\beta=0$, $y=\hat x$.
- **Where.** Stabilizes the residual stream before attention and before the MLP; modern decoders use **pre-norm**, normalizing the sublayer input.
- **Triton.** One program per row: one pass for the sum and one for the sum of squares (or Welford), then `tl.rsqrt(var + eps)` and the affine map.
- **Failure.** $\epsilon$ outside the square root; dividing by $d-1$ instead of $d$; forgetting the affine parameters; normalizing across the batch axis.
- **Bridge.** §23.10 removes the mean subtraction and the bias.


### 23.10 RMSNorm

- **Connection.** Builds on §23.9: same per-row normalization, but without mean subtraction or bias.
- **Rule.** Scale each row by the reciprocal root-mean-square and apply $\gamma$; same shape as the input.
- **Formula.** $y_j=\gamma_j x_j/\sqrt{\frac1d\sum_k x_k^2+\epsilon}$.
- **Example.** For $x=(1,2,3,4)$:

$$
\frac1d\sum_k x_k^2=7.5,\qquad
\operatorname{rsqrt}(7.5+10^{-6})=0.3651,\qquad
y=(0.3651,\;0.7303,\;1.0954,\;1.4606).
$$

- **Where.** Cheaper than LayerNorm (one reduction instead of two); used by Llama-family models. Dropping the mean is a modeling choice, not an approximation.
- **Triton.** Reduce `x*x`, scale by `tl.rsqrt(mean + eps)`, multiply by $\gamma$; one load, one reduction, one store.
- **Failure.** Reusing a LayerNorm kernel that still applies `beta`; $\epsilon$ outside the root; accumulating the sum of squares in FP16.
- **Bridge.** §23.11 feeds the normalized rows into the MLP.


### 23.11 MLP

- **Connection.** Builds on §23.9–§23.10: normalized rows now get channel mixing, the other half of the block.
- **Rule.** Two linear maps with GELU between them; hidden width $4d$, so $[T,d]\to[T,4d]\to[T,d]$.
- **Formula.** $Y=\operatorname{GELU}(XW_1+b_1)W_2+b_2$ with $W_1\in\mathbb R^{d\times 4d}$ and $W_2\in\mathbb R^{4d\times d}$.
- **Example.** For $x=(1,1)$:

$$
W_1=\begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix},\quad
xW_1=(1,1,0,0),\quad
\operatorname{GELU}(1,1,0,0)=(0.8413,0.8413,0,0),
$$

  - **Output.** With $W_2=\begin{bmatrix}1&0\\0&1\\0&0\\0&0\end{bmatrix}$, $y=(0.8413,0.8413)$.
- **Where.** Attention mixes tokens; the MLP mixes channels. The expansion factor $4$ gives the nonlinearity room before projecting back; GELU and SiLU are common.
- **Triton.** Two `tl.dot` operations with a GELU epilogue: `pre = dot(x,w1)+b1`, `h = GELU(pre)`, `y = dot(h,w2)+b2`.
- **Failure.** The wrong activation changes every value; broadcasting the bias on the wrong axis; transposing $W_2$ is a shape error when $4d\ne d$ and silently wrong when they coincide.
- **Bridge.** §23.12 composes attention and MLP into one residual block.


### 23.12 Fused transformer blocks

- **Connection.** Builds on §23.1–§23.11: the composition of every previous stage around two residual adds.
- **Rule.** Pre-norm block: every sublayer maps $[B,T,d]\to[B,T,d]$, so the residual add is always legal; the block changes neither sequence length nor width.
- **Formula.** $Y=X+\operatorname{MLP}\left(\operatorname{LN}\left(X+\operatorname{Attn}(\operatorname{LN}(X))\right)\right)$.
- **Example.** With $X$ and $O$ from §23.4–§23.7, the first residual add is:

$$
H_1=X+O=
\begin{bmatrix}1&1\\1&0\end{bmatrix}
+
\begin{bmatrix}1.0000&1.3395\\1.0000&1.0000\end{bmatrix}
=
\begin{bmatrix}2.0000&2.3395\\2.0000&1.0000\end{bmatrix}.
$$

  - **Second add.** The second residual add has the same shape; the block output is again $[T,d]$.
- **Where.** Fusion removes intermediate tensors between LN, QKV, attention, and the MLP; a production block keeps the residual stream in registers where possible.
- **Triton.** Fuse LN + QKV, then the score/softmax/output pipeline, then LN + MLP; the residual add is a vector add folded into the branch epilogue.
- **Failure.** Post-norm instead of pre-norm; dropping a residual; applying the second LN to $X$ instead of $H_1$; fusing across a dependency that forces recomputation.
- **Bridge.** §23.13 moves from training to inference and caches K/V.


### 23.13 KV cache

- **Connection.** Builds on §23.4–§23.8: the same K/V tensors, now reused across decode steps instead of recomputed.
- **Rule.** Append one K/V row at position $t$ and read the whole prefix; the cache grows $[B,H,T,d]\to[B,H,T+1,d]$, the query is $[B,H,1,d]$, and the scores are $[B,H,1,t]$.
- **Formula.** $K^{\text{cache}}_{b,h,t,:}\leftarrow k_{b,h,:}$, $V^{\text{cache}}_{b,h,t,:}\leftarrow v_{b,h,:}$, and $S_{b,h,1,:}=q_{b,h,:}K^{\text{cache}\top}_{b,h,:,:}$.
- **Example.** With $B=2$, $H=4$, $T=3$, $d=2$:

| object | shape | elements | FP16 bytes |
|---|---|---:|---:|
| $K^{\text{cache}}$ before | $(2,4,3,2)$ | 48 | 96 |
| new $k$ | $(2,4,1,2)$ | 16 | 32 |
| $K^{\text{cache}}$ after | $(2,4,4,2)$ | 64 | 128 |
| $Q$ at decode | $(2,4,1,2)$ | 16 | 32 |
| scores | $(2,4,1,4)$ | 32 | 64 |
| output | $(2,4,1,2)$ | 16 | 32 |

- **Where.** Autoregressive decoding would recompute all past K/V; the cache stores them once and reads them at every token, and that read dominates long-context inference.
- **Triton.** Use the address `((b * H + h) * T + pos) * d + c` for a contiguous $[B,H,T,d]$ cache; pass the stride explicitly for a paged or transposed cache.
- **Failure.** Assuming contiguity under a page table; writing the wrong position; accumulating in FP16; forgetting that the cache must grow or that a fixed buffer must be masked.
- **Bridge.** The worked example instantiates this stage next, then Part II builds the reference.

**Where this leads.**
- **Worked example.** A two-token, $d=2$ sequence makes every shape and value concrete.
- **Part II.** The same stages become readable PyTorch functions and CPU assertions.
- **Part III.** The stages become masked Triton kernels with `run_if_gpu` tests.
- **Part IV.** Attention and MLP are fused; the KV-cache section handles inference traffic.


## Chapter map

| § | Stage | Formula | Shape rule | Kernel/demo |
|---|---|---|---|---|
| **23.1** | Embedding | $Y_i=E_{\pi(i),:}$ | $(\cdots)\to(\cdots,d)$ | gather kernel |
| **23.2** | Position encoding | $PE_{t,2i}=\sin(t/10000^{2i/d})$ | $[T,d]\to[T,d]$ | `position_ref` + formula |
| **23.3** | RoPE | $x'=R(p)x$ | $[T,d]\to[T,d]$ | `rope_ref` + formula |
| **23.4** | QKV projection | $(Q,K,V)=X(W_q,W_k,W_v)$ | $[T,d]\to[T,3d]$ | `qkv_ref` projection |
| **23.5** | Attention scores | $S=QK^\top/\sqrt d$ | $[T_q,d]\times[T_k,d]\to[T_q,T_k]$ | score kernel |
| **23.6** | Softmax | $P_{ij}=e^{S_{ij}-m_i}/\sum_j e^{S_{ij}-m_i}$ | $[T_q,T_k]\to[T_q,T_k]$ | row-softmax kernel |
| **23.7** | Attention output | $O=PV$ | $[T_q,T_k]\times[T_k,d]\to[T_q,d]$ | output kernel |
| **23.8** | FlashAttention | online $(m,\ell,\mathrm{acc})$ | state $[T_q]$ and $[T_q,d]$ | two-tile flash kernel |
| **23.9** | LayerNorm | $y=\gamma(x-\mu)/\sqrt{\sigma^2+\epsilon}+\beta$ | $[T,d]\to[T,d]$ | norm kernel |
| **23.10** | RMSNorm | $y=\gamma x/\sqrt{\operatorname{mean}(x^2)+\epsilon}$ | $[T,d]\to[T,d]$ | norm kernel (`RMS=True`) |
| **23.11** | MLP | $Y=\operatorname{GELU}(XW_1+b_1)W_2+b_2$ | $[T,d]\to[T,4d]\to[T,d]$ | fused MLP kernel |
| **23.12** | Fused blocks | $Y=X+\operatorname{MLP}(\operatorname{LN}(X+\operatorname{Attn}(\operatorname{LN}(X))))$ | $[T,d]\to[T,d]$ | `block_ref` composition |
| **23.13** | KV cache | $K^{\text{cache}}_{b,h,t,:}\leftarrow k$ | $[B,H,T,d]\to[B,H,T+1,d]$ | append kernel |

- **Square object.** The score matrix is $T_q\times T_k$, not $d\times d$.
- **Everything else.** A row-wise map or a projection back to width $d$.


In [ ]:
# Every shape claim in the chapter map, checked before any kernel is written.
B, T, H, D = 2, 2, 2, 2
d = H * D
X = torch.randn(B, T, d)

Q = X @ torch.randn(d, d)
K = X @ torch.randn(d, d)
V = X @ torch.randn(d, d)
assert Q.shape == K.shape == V.shape == (B, T, d)

# [B,T,d] -> [B,H,T,D] -> scores [B,H,T,T] -> output [B,H,T,D]
Qh = Q.view(B, T, H, D).transpose(1, 2)
Kh = K.view(B, T, H, D).transpose(1, 2)
Vh = V.view(B, T, H, D).transpose(1, 2)
assert Qh.shape == Kh.shape == Vh.shape == (B, H, T, D)

S = Qh @ Kh.transpose(-2, -1) / math.sqrt(D)
P = torch.softmax(S, dim=-1)
O = P @ Vh
assert S.shape == P.shape == (B, H, T, T)
assert O.shape == (B, H, T, D)
assert torch.allclose(P.sum(-1), torch.ones(B, H, T))

# The causal mask keeps the lower triangle and zeroes the strict upper one.
mask = torch.ones(T, T, dtype=torch.bool).tril()
S_causal = S.masked_fill(~mask, float("-inf"))
P_causal = torch.softmax(S_causal, dim=-1)
assert torch.equal(P_causal[..., mask.logical_not()], torch.zeros_like(P_causal[..., mask.logical_not()]))

# KV cache grows along the time axis, not the head or channel axis.
cache = torch.zeros(B, H, T, D)
assert cache.shape == (2, 2, 2, 2)
assert cache.numel() == B * H * T * D


In [ ]:
# Head layout and the causal tile map, checked with assertions.
B, T, H, D = 2, 3, 2, 2
X = torch.arange(B * T * H * D, dtype=torch.float32).view(B, T, H, D)
Xh = X.transpose(1, 2)
assert Xh.shape == (B, H, T, D)
assert Xh.stride() == (T * H * D, D, H * D, 1)

# The transpose moves the head axis; it does not reorder any values.
for b in range(B):
    for h in range(H):
        for t in range(T):
            assert torch.equal(Xh[b, h, t], X[b, t, h])

# Two K/V tiles of width 16 cover T=32.
T2, BN = 32, 16
tiles = [(start, start + BN - 1) for start in range(0, T2, BN)]
assert tiles == [(0, 15), (16, 31)]
assert sum(stop - start + 1 for start, stop in tiles) == T2

# Causal mask: row 0 keeps only key 0; row 31 keeps keys 16..31 in tile 2.
assert [k for k in range(0, 16) if k <= 0] == [0]
assert [k for k in range(16, 32) if k <= 31] == list(range(16, 32))


## Worked numerical example

- **Spine.** A two-token sequence with $d=2$; every number below is checked by the next code cell with `torch.allclose`.
- **No prints.** The worked-example cell is asserts only.

### Embedding

- **Table and ids.** The lookup copies rows 2 and 0.

$$
E=\begin{bmatrix}1&0\\0&1\\1&1\\2&-1\end{bmatrix},
\qquad
\pi=(2,0),
\qquad
X=E[\pi]=\begin{bmatrix}1&1\\1&0\end{bmatrix}.
$$

### QKV projection

- **Weights.** $W_q=W_k=I$ and $W_v=\begin{bmatrix}1&0\\0&2\end{bmatrix}$.

$$
Q=\begin{bmatrix}1&1\\1&0\end{bmatrix},\qquad
K=\begin{bmatrix}1&1\\1&0\end{bmatrix},\qquad
V=\begin{bmatrix}1&2\\1&0\end{bmatrix}.
$$

### Scores, softmax, output

- **Pipeline.** $S=QK^\top/\sqrt2$, then row softmax, then $O=PV$.

$$
\begin{aligned}
S&=\frac{QK^\top}{\sqrt2}
=\begin{bmatrix}1.4142&0.7071\\0.7071&0.7071\end{bmatrix},\\
P&=\operatorname{softmax}(S)
=\begin{bmatrix}0.6698&0.3302\\0.5&0.5\end{bmatrix},\\
O&=PV
=\begin{bmatrix}1.0000&1.3395\\1.0000&1.0000\end{bmatrix}.
\end{aligned}
$$

- **Hand checks.** $S_{00}=(1\cdot1+1\cdot1)/\sqrt2=\sqrt2=1.4142$; $e^{1.4142}/(e^{1.4142}+e^{0.7071})=0.6698$; each row of $P$ sums to one; $O_{01}=0.6698\cdot2+0.3302\cdot0=1.3395$.


### One RoPE rotation

- **Setup.** First pair of $x=(1,0)$ at position $p=1$: $\theta_0=1$.

$$
\theta_0=1\cdot10000^{-0/2}=1,
\qquad
\begin{bmatrix}x'_{0}\\x'_{1}\end{bmatrix}
=
\begin{bmatrix}\cos1&-\sin1\\ \sin1&\cos1\end{bmatrix}
\begin{bmatrix}1\\0\end{bmatrix}
=
\begin{bmatrix}0.5403\\0.8415\end{bmatrix}.
$$

### KV-cache shapes

- **Counts.** For $B=2$, $H=4$, $T=3$ past tokens, $d=2$:

| object | shape | elements | FP16 bytes |
|---|---|---:|---:|
| $K$ cache before | $(2,4,3,2)$ | $48$ | $96$ |
| new $k$ at $t=3$ | $(2,4,1,2)$ | $16$ | $32$ |
| $K$ cache after | $(2,4,4,2)$ | $64$ | $128$ |
| scores against cache | $(2,4,1,4)$ | $32$ | $64$ |

- **Arithmetic.** $B\cdot H\cdot T\cdot d=2\cdot4\cdot3\cdot2=48$ elements and $48\cdot2=96$ bytes in FP16; after the append $T=4$ and the cache is $64$ elements, $128$ bytes.

### FlashAttention trace for row 0

| tile | $m$ | $\ell$ | $\mathrm{acc}$ |
|---:|---:|---:|---|
| 1: key 0 | $1.4142$ | $1.0000$ | $(1.0000,2.0000)$ |
| 2: key 1 | $1.4142$ | $1.4931$ | $(1.4931,2.0000)$ |

- **Result.** The final division gives $(1.0000,1.3395)$, matching $O_0$ above.


In [ ]:
def test_worked_example():
    '''Every number in the worked example, checked exactly.'''
    E = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, -1.0]])
    ids = torch.tensor([2, 0])
    X = E[ids]
    assert X.shape == (2, 2)
    assert torch.equal(X, torch.tensor([[1.0, 1.0], [1.0, 0.0]]))

    Wq = torch.eye(2)
    Wk = torch.eye(2)
    Wv = torch.tensor([[1.0, 0.0], [0.0, 2.0]])
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    assert torch.equal(Q, torch.tensor([[1.0, 1.0], [1.0, 0.0]]))
    assert torch.equal(K, torch.tensor([[1.0, 1.0], [1.0, 0.0]]))
    assert torch.equal(V, torch.tensor([[1.0, 2.0], [1.0, 0.0]]))

    S = Q @ K.T / math.sqrt(2.0)
    S_expected = torch.tensor([[1.41421356, 0.70710678], [0.70710678, 0.70710678]])
    assert torch.allclose(S, S_expected, atol=1e-6)

    P = torch.softmax(S, dim=1)
    P_expected = torch.tensor([[0.6698, 0.3302], [0.5, 0.5]])
    assert torch.allclose(P, P_expected, atol=1e-4)
    assert torch.allclose(P.sum(1), torch.ones(2), atol=1e-6)

    O = P @ V
    O_expected = torch.tensor([[1.0, 1.3395], [1.0, 1.0]])
    assert torch.allclose(O, O_expected, atol=1e-4)

    # One RoPE rotation: x=(1,0), p=1, theta=1.
    x = torch.tensor([1.0, 0.0])
    c, s = math.cos(1.0), math.sin(1.0)
    rope = torch.stack([x[0] * c - x[1] * s, x[0] * s + x[1] * c])
    assert torch.allclose(rope, torch.tensor([0.5403, 0.8415]), atol=1e-4)

    # KV-cache shape and byte table.
    B, H, T, D = 2, 4, 3, 2
    cache = torch.zeros(B, H, T, D, dtype=torch.float16)
    new_k = torch.zeros(B, H, 1, D, dtype=torch.float16)
    assert cache.shape == (2, 4, 3, 2)
    assert cache.numel() == B * H * T * D == 48
    assert cache.numel() * cache.element_size() == 96
    assert new_k.numel() == B * H * D == 16
    assert new_k.numel() * new_k.element_size() == 32
    assert (B, H, T + 1, D) == (2, 4, 4, 2)
    assert B * H * (T + 1) * D == 64
    assert B * H * (T + 1) * D * 2 == 128


test_worked_example()


## Part II — Method 01: the stage-by-stage PyTorch reference

- **Purpose.** A slow, readable oracle; each function is one line of the formula and runs on CPU.
- **Coverage.** The two code cells cover, in order:
  - embedding, position encoding, RoPE;
  - QKV projection, attention scores, softmax, output;
  - LayerNorm, RMSNorm, GELU, MLP, the full block, and the KV-cache append.
- **Tolerances.** Exact where values are exact (embedding, QKV, residuals); `torch.allclose` where a transcendental function is involved (softmax, sin/cos, GELU).
- **Why it matters.** If Method 02 disagrees with Method 01 on the spine, the kernel is wrong.


In [ ]:
def embedding_ref(table, ids):
    return table[ids]


def position_ref(T, D, base=10000.0):
    pos = torch.arange(T, dtype=torch.float32)[:, None]
    i = torch.arange(D, dtype=torch.float32)[None, :]
    pair = (i // 2).long()
    angle = pos * base ** (-2.0 * pair / D)
    return torch.where(i.long() % 2 == 0, torch.sin(angle), torch.cos(angle))


def rope_ref(x, pos, base=10000.0):
    x = torch.as_tensor(x, dtype=torch.float32)
    D = x.shape[-1]
    i = torch.arange(D // 2, dtype=torch.float32)
    theta = float(pos) * base ** (-2.0 * i / D)
    c, s = torch.cos(theta), torch.sin(theta)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * c - x2 * s
    out[..., 1::2] = x1 * s + x2 * c
    return out


def qkv_ref(X, Wq, Wk, Wv):
    return X @ Wq, X @ Wk, X @ Wv


def attention_ref(Q, K, V, causal=False):
    d = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / math.sqrt(d)
    if causal:
        Tq, Tk = S.shape[-2:]
        mask = torch.ones(Tq, Tk, dtype=torch.bool).tril()
        S = S.masked_fill(~mask, float("-inf"))
    P = torch.softmax(S, dim=-1)
    return P @ V, P, S


def test_method01_attention():
    E = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, -1.0]])
    ids = torch.tensor([2, 0])
    X = embedding_ref(E, ids)
    assert torch.equal(X, torch.tensor([[1.0, 1.0], [1.0, 0.0]]))

    pe = position_ref(2, 4)
    pe_expected = torch.tensor([
        [0.0, 1.0, 0.0, 1.0],
        [0.84147096, 0.54030228, 0.00999983, 0.99995000],
    ])
    assert torch.allclose(pe, pe_expected, atol=1e-6)

    assert torch.allclose(
        rope_ref(torch.tensor([1.0, 0.0]), 1),
        torch.tensor([0.54030231, 0.84147098]),
        atol=1e-6,
    )

    Wq, Wk = torch.eye(2), torch.eye(2)
    Wv = torch.tensor([[1.0, 0.0], [0.0, 2.0]])
    Q, K, V = qkv_ref(X, Wq, Wk, Wv)
    O, P, S = attention_ref(Q, K, V)
    assert torch.allclose(S, torch.tensor([[1.41421356, 0.70710678], [0.70710678, 0.70710678]]), atol=1e-6)
    assert torch.allclose(P, torch.tensor([[0.6698, 0.3302], [0.5, 0.5]]), atol=1e-4)
    assert torch.allclose(O, torch.tensor([[1.0, 1.3395], [1.0, 1.0]]), atol=1e-4)

    # Causal attention: row 0 sees only key 0, so its output is V[0].
    O_c, P_c, _ = attention_ref(Q, K, V, causal=True)
    assert torch.allclose(P_c[0], torch.tensor([1.0, 0.0]), atol=1e-6)
    assert torch.allclose(O_c[0], V[0], atol=1e-6)
    assert P_c[0, 1].item() == 0.0


test_method01_attention()
print("method 01: embedding, position, RoPE, QKV, and attention match")


In [ ]:
def layernorm_ref(x, gamma, beta, eps=1e-5):
    mu = x.mean(dim=-1, keepdim=True)
    var = ((x - mu) ** 2).mean(dim=-1, keepdim=True)
    return (x - mu) / torch.sqrt(var + eps) * gamma + beta


def rmsnorm_ref(x, gamma, eps=1e-6):
    return x * torch.rsqrt((x * x).mean(dim=-1, keepdim=True) + eps) * gamma


def gelu_ref(x):
    return 0.5 * x * (1.0 + torch.erf(x / math.sqrt(2.0)))


def mlp_ref(X, W1, b1, W2, b2):
    return gelu_ref(X @ W1 + b1) @ W2 + b2


def block_ref(X, Wq, Wk, Wv, Wo, W1, b1, W2, b2, g1, beta1, g2, beta2):
    h = X + attention_ref(layernorm_ref(X, g1, beta1) @ Wq,
                          layernorm_ref(X, g1, beta1) @ Wk,
                          layernorm_ref(X, g1, beta1) @ Wv)[0] @ Wo
    y = h + mlp_ref(layernorm_ref(h, g2, beta2), W1, b1, W2, b2)
    return y


def kv_append_ref(k_cache, v_cache, k_new, v_new, pos):
    k_cache[:, :, pos:pos + 1, :] = k_new
    v_cache[:, :, pos:pos + 1, :] = v_new
    return k_cache, v_cache


def test_method01_norms_mlp_block():
    x = torch.tensor([1.0, 2.0, 3.0, 4.0])
    gamma = torch.ones(4)
    beta = torch.zeros(4)
    ln = layernorm_ref(x, gamma, beta)
    assert torch.allclose(ln, torch.tensor([-1.3416, -0.4472, 0.4472, 1.3416]), atol=1e-4)
    assert abs(ln.sum().item()) < 1e-6

    rms = rmsnorm_ref(x, gamma)
    assert torch.allclose(rms, torch.tensor([0.3651, 0.7303, 1.0954, 1.4606]), atol=1e-4)

    # MLP tiny example: x=(1,1), hidden pre-activation (1,1,0,0).
    xm = torch.tensor([[1.0, 1.0]])
    W1 = torch.tensor([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
    W2 = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0], [0.0, 0.0]])
    b1 = torch.zeros(4)
    b2 = torch.zeros(2)
    y = mlp_ref(xm, W1, b1, W2, b2)
    assert torch.allclose(y, torch.tensor([[0.8413, 0.8413]]), atol=1e-4)

    # Residual identity: zero value/output weights make the block the identity.
    X = torch.tensor([[1.0, 1.0], [1.0, 0.0]])
    Wq = Wk = torch.eye(2)
    Wv = torch.zeros(2, 2)
    Wo = torch.eye(2)
    W1b = torch.zeros(2, 4)
    W2b = torch.zeros(4, 2)
    g = torch.ones(2)
    zero4 = torch.zeros(4)
    zero2 = torch.zeros(2)
    Y = block_ref(X, Wq, Wk, Wv, Wo, W1b, zero4, W2b, zero2, g, zero2, g, zero2)
    assert torch.allclose(Y, X, atol=1e-6)

    # KV-cache append writes exactly one position.
    k_cache = torch.zeros(2, 4, 4, 2)
    v_cache = torch.zeros(2, 4, 4, 2)
    k_new = torch.ones(2, 4, 1, 2)
    v_new = torch.full((2, 4, 1, 2), 2.0)
    kv_append_ref(k_cache, v_cache, k_new, v_new, pos=2)
    assert torch.equal(k_cache[:, :, 2, :], torch.ones(2, 4, 2))
    assert torch.equal(v_cache[:, :, 2, :], torch.full((2, 4, 2), 2.0))
    assert torch.equal(k_cache[:, :, 0, :], torch.zeros(2, 4, 2))
    assert k_cache.shape == (2, 4, 4, 2)


test_method01_norms_mlp_block()
print("method 01: norms, MLP, block, and KV append match")


## Part III — Method 02: one Triton kernel per stage

- **Same contract.** Each kernel mirrors its Method 01 function.
- **Host wrapper.** Allocates the output and chooses the grid.
- **Masking.** Every load and store is guarded.
- **Silence.** The kernel never prints and never synchronizes.
- **Test.** A silent `assert` against PyTorch, collected with `run_if_gpu`.
- **Tile map.** For a 2-D grid, $\text{rows}=p_mB_M+\{0,\dots,B_M-1\}$ and $\text{cols}=p_nB_N+\{0,\dots,B_N-1\}$ with $p_m=\texttt{tl.program\_id}(0)$ and $p_n=\texttt{tl.program\_id}(1)$.
- **Tail.** Programs in the tail are masked, so the same kernel handles odd sizes.
- **Dot sizes.** GPU tests use $T=32$, $D=16$ because `tl.dot` expects tiles of at least $16$; the tiny $d=2$ example is the hand-check.


In [ ]:
@triton.jit
def embedding_kernel(table_ptr, ids_ptr, out_ptr, n_tokens, dim,
                     BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    row_mask = rows < n_tokens
    col_mask = cols < dim
    idx = tl.load(ids_ptr + rows, mask=row_mask, other=0)
    keep = row_mask[:, None] & col_mask[None, :]
    value = tl.load(table_ptr + idx[:, None] * dim + cols[None, :],
                    mask=keep, other=0.0)
    tl.store(out_ptr + rows[:, None] * dim + cols[None, :], value, mask=keep)


def embedding_triton(table, ids, BLOCK_M=4, BLOCK_N=8):
    table = table.contiguous()
    ids = ids.contiguous()
    out = torch.empty(*ids.shape, table.shape[1], device=table.device, dtype=table.dtype)
    n_tokens = ids.numel()
    grid = (triton.cdiv(n_tokens, BLOCK_M), triton.cdiv(table.shape[1], BLOCK_N))
    embedding_kernel[grid](table, ids.to(torch.int32), out, n_tokens, table.shape[1],
                           BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_embedding_kernel():
    torch.manual_seed(0)
    table = torch.randn(37, 19, device=DEVICE)
    ids = torch.randint(0, 37, (5, 7), device=DEVICE)
    out = embedding_triton(table, ids)
    assert out.shape == (5, 7, 19)
    assert torch.allclose(out, table[ids], atol=1e-6, rtol=1e-6)


run_if_gpu(test_embedding_kernel)


In [ ]:
@triton.jit
def norm_kernel(x_ptr, w_ptr, b_ptr, out_ptr, M, N, eps,
                RMS: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    c = tl.arange(0, BLOCK_N)
    mask = (row < M) & (c < N)
    x = tl.load(x_ptr + row * N + c, mask=mask, other=0.0).to(tl.float32)

    if RMS:
        centered = x
    else:
        mean = tl.sum(tl.where(mask, x, 0.0), axis=0) / N
        centered = tl.where(mask, x - mean, 0.0)

    var = tl.sum(centered * centered, axis=0) / N
    inv = tl.rsqrt(var + eps)
    w = tl.load(w_ptr + c, mask=c < N, other=0.0).to(tl.float32)

    if RMS:
        y = centered * inv * w
    else:
        b = tl.load(b_ptr + c, mask=c < N, other=0.0).to(tl.float32)
        y = centered * inv * w + b

    tl.store(out_ptr + row * N + c, y, mask=mask)


def norm_triton(x, gamma, beta=None, eps=1e-5, RMS=False, BLOCK_N=None):
    x = x.contiguous()
    M, N = x.shape
    BLOCK_N = BLOCK_N or triton.next_power_of_2(N)
    out = torch.empty_like(x)
    b = gamma if beta is None else beta
    norm_kernel[(M,)](x, gamma, b, out, M, N, eps, RMS=RMS, BLOCK_N=BLOCK_N)
    return out


def test_norm_kernel():
    torch.manual_seed(0)
    x = torch.randn(5, 7, device=DEVICE)
    gamma = torch.randn(7, device=DEVICE)
    beta = torch.randn(7, device=DEVICE)

    y_ln = norm_triton(x, gamma, beta, eps=1e-5, RMS=False)
    assert torch.allclose(y_ln, layernorm_ref(x, gamma, beta, 1e-5), atol=1e-5, rtol=1e-5)

    y_rms = norm_triton(x, gamma, eps=1e-6, RMS=True)
    assert torch.allclose(y_rms, rmsnorm_ref(x, gamma, 1e-6), atol=1e-5, rtol=1e-5)


run_if_gpu(test_norm_kernel)


In [ ]:
@triton.jit
def row_softmax_kernel(x_ptr, out_ptr, M, N, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    c = tl.arange(0, BLOCK_N)
    mask = (row < M) & (c < N)
    x = tl.load(x_ptr + row * N + c, mask=mask, other=-float("inf"))
    m = tl.max(x, axis=0)
    e = tl.exp(x - m)
    total = tl.sum(e, axis=0)
    tl.store(out_ptr + row * N + c, e / total, mask=mask)


def row_softmax_triton(x, BLOCK_N=None):
    x = x.contiguous()
    M, N = x.shape
    BLOCK_N = BLOCK_N or triton.next_power_of_2(N)
    out = torch.empty_like(x)
    row_softmax_kernel[(M,)](x, out, M, N, BLOCK_N=BLOCK_N)
    return out


def test_row_softmax():
    torch.manual_seed(0)
    x = torch.randn(5, 7, device=DEVICE)
    out = row_softmax_triton(x)
    assert torch.allclose(out, torch.softmax(x, dim=1), atol=1e-6, rtol=1e-6)
    assert torch.allclose(out.sum(1), torch.ones(5, device=DEVICE), atol=1e-6)


run_if_gpu(test_row_softmax)


In [ ]:
@triton.jit
def attention_scores_kernel(q_ptr, k_ptr, s_ptr, T, D: tl.constexpr,
                            BM: tl.constexpr, BN: tl.constexpr, scale):
    pm = tl.program_id(0)
    pn = tl.program_id(1)
    rows = pm * BM + tl.arange(0, BM)
    cols = pn * BN + tl.arange(0, BN)
    d = tl.arange(0, D)

    q = tl.load(q_ptr + rows[:, None] * D + d[None, :],
                mask=rows[:, None] < T, other=0.0)
    k = tl.load(k_ptr + cols[:, None] * D + d[None, :],
                mask=cols[:, None] < T, other=0.0)
    s = tl.dot(q, tl.trans(k), input_precision="ieee") * scale
    keep = (rows[:, None] < T) & (cols[None, :] < T)
    tl.store(s_ptr + rows[:, None] * T + cols[None, :], s, mask=keep)


def attention_scores_triton(Q, K, BM=16, BN=16):
    Q, K = Q.contiguous(), K.contiguous()
    T, D = Q.shape
    S = torch.empty(T, T, device=Q.device, dtype=torch.float32)
    grid = (triton.cdiv(T, BM), triton.cdiv(T, BN))
    attention_scores_kernel[grid](Q, K, S, T, D=D, BM=BM, BN=BN,
                                  scale=1.0 / math.sqrt(D))
    return S


def test_attention_scores():
    torch.manual_seed(0)
    T, D = 32, 16
    Q = torch.randn(T, D, device=DEVICE)
    K = torch.randn(T, D, device=DEVICE)
    S = attention_scores_triton(Q, K)
    ref = Q @ K.T / math.sqrt(D)
    assert S.shape == (T, T)
    assert torch.allclose(S, ref, atol=1e-5, rtol=1e-5)


run_if_gpu(test_attention_scores)


In [ ]:
@triton.jit
def attention_output_kernel(p_ptr, v_ptr, o_ptr, T, D: tl.constexpr,
                            BM: tl.constexpr, BK: tl.constexpr):
    pm = tl.program_id(0)
    rows = pm * BM + tl.arange(0, BM)
    keys = tl.arange(0, BK)
    d = tl.arange(0, D)

    p = tl.load(p_ptr + rows[:, None] * T + keys[None, :],
                mask=(rows[:, None] < T) & (keys[None, :] < T), other=0.0)
    v = tl.load(v_ptr + keys[:, None] * D + d[None, :],
                mask=keys[:, None] < T, other=0.0)
    o = tl.dot(p, v, input_precision="ieee")
    tl.store(o_ptr + rows[:, None] * D + d[None, :], o, mask=rows[:, None] < T)


def attention_output_triton(P, V, BM=16, BK=32):
    P, V = P.contiguous(), V.contiguous()
    T, D = V.shape
    O = torch.empty(T, D, device=V.device, dtype=torch.float32)
    grid = (triton.cdiv(T, BM),)
    attention_output_kernel[grid](P, V, O, T, D=D, BM=BM, BK=BK)
    return O


def test_attention_output():
    torch.manual_seed(0)
    T, D = 32, 16
    S = torch.randn(T, T, device=DEVICE)
    P = torch.softmax(S, dim=1)
    V = torch.randn(T, D, device=DEVICE)
    O = attention_output_triton(P, V)
    assert O.shape == (T, D)
    assert torch.allclose(O, P @ V, atol=1e-5, rtol=1e-5)


run_if_gpu(test_attention_output)


## Cost model: FLOPs, bytes, and the square

- **Arithmetic.** Scores cost $2T^2d$ FLOPs and the output costs $2T^2d$, so $4T^2d$ total per head.
- **Direct traffic.** Writing and re-reading $S$ and $P$ costs about $Q_{\text{direct}}\approx 4T^2b$ bytes.
- **Flash traffic.** Keeping the row state and streaming K/V costs about $Q_{\text{flash}}\approx 4Tdb$ bytes.
- **Ratio.** $Q_{\text{direct}}/Q_{\text{flash}}=T/d$.
- **Concrete.** For $T=4096$, $d=128$: the score matrix has $16{,}777{,}216$ elements, the state has $524{,}288$, and the ratio is $32$.
- **Takeaway.** The arithmetic is unchanged; only the traffic moves, which is why FlashAttention is a kernel-level optimization rather than an approximation.


## Part IV — Method 03: online softmax and block fusion

- **Direct path.** Materializes $S$ and $P$, each $T\times T$.
- **Flash path.** Streams K/V tiles and keeps only the row state.
- **Merge.** Running state $(m,\ell,\mathrm{acc})$ and a new tile $(m_{\text{tile}},\ell_{\text{tile}},\mathrm{acc}_{\text{tile}})$ combine as:

$$
\begin{aligned}
m'&=\max(m,m_{\text{tile}}),\\
\ell'&=e^{m-m'}\ell+e^{m_{\text{tile}}-m'}\ell_{\text{tile}},\\
\mathrm{acc}'&=e^{m-m'}\mathrm{acc}+e^{m_{\text{tile}}-m'}\mathrm{acc}_{\text{tile}}.
\end{aligned}
$$

- **Exactness.** The correction factors are the same for every entry of the row, so this is exact, not an approximation.
- **Scaling.** The kernel performs $O(T^2d)$ arithmetic but stores only $O(Td)$ state; the score tile $B_M\times B_N$ is discarded.
- **Concrete.** For $T=4096$, $d=128$: the direct score matrix is $16$M elements per head; the FlashAttention state is $4096\times128=0.5$M elements, and the $T\times T$ matrix never exists.
- **Test shape.** $T=32$, $D=16$, $B_N=16$ gives exactly two K/V tiles, the smallest setting that exercises rescaling with `tl.dot`.


In [ ]:
def test_cost_model():
    T, d, b = 4096, 128, 2
    score_elems = T * T
    state_elems = T * d
    assert score_elems == 16_777_216
    assert state_elems == 524_288
    assert score_elems // state_elems == 32

    flops = 4 * T * T * d
    assert flops == 8_589_934_592

    direct_bytes = 4 * T * T * b
    flash_bytes = 4 * T * d * b
    assert direct_bytes // flash_bytes == T // d == 32


test_cost_model()
print("cost model: 32x less score state, 8.6 GFLOP per head at T=4096")


In [ ]:
@triton.jit
def flash_attention_kernel(q_ptr, k_ptr, v_ptr, o_ptr, T,
                           D: tl.constexpr, BM: tl.constexpr, BN: tl.constexpr,
                           scale, CAUSAL: tl.constexpr):
    pid = tl.program_id(0)
    rows = pid * BM + tl.arange(0, BM)
    d = tl.arange(0, D)

    q = tl.load(q_ptr + rows[:, None] * D + d[None, :],
                mask=rows[:, None] < T, other=0.0)
    m = tl.full((BM,), -float("inf"), tl.float32)
    l = tl.zeros((BM,), tl.float32)
    acc = tl.zeros((BM, D), tl.float32)

    for start in range(0, T, BN):
        cols = start + tl.arange(0, BN)
        k = tl.load(k_ptr + cols[:, None] * D + d[None, :],
                    mask=cols[:, None] < T, other=0.0)
        v = tl.load(v_ptr + cols[:, None] * D + d[None, :],
                    mask=cols[:, None] < T, other=0.0)

        s = tl.dot(q, tl.trans(k), input_precision="ieee") * scale
        if CAUSAL:
            s = tl.where(cols[None, :] <= rows[:, None], s, -float("inf"))
        s = tl.where(cols[None, :] < T, s, -float("inf"))

        m_new = tl.maximum(m, tl.max(s, axis=1))
        p = tl.exp(s - m_new[:, None])
        alpha = tl.exp(m - m_new)
        l = l * alpha + tl.sum(p, axis=1)
        acc = acc * alpha[:, None] + tl.dot(p, v, input_precision="ieee")
        m = m_new

    o = acc / l[:, None]
    tl.store(o_ptr + rows[:, None] * D + d[None, :], o, mask=rows[:, None] < T)


def flash_attention_triton(Q, K, V, BM=16, BN=16, causal=False):
    Q, K, V = Q.contiguous(), K.contiguous(), V.contiguous()
    T, D = Q.shape
    O = torch.empty(T, D, device=Q.device, dtype=torch.float32)
    grid = (triton.cdiv(T, BM),)
    flash_attention_kernel[grid](Q, K, V, O, T, D=D, BM=BM, BN=BN,
                                 scale=1.0 / math.sqrt(D), CAUSAL=causal)
    return O


def causal_reference(Q, K, V):
    T, D = Q.shape
    S = Q @ K.T / math.sqrt(D)
    mask = torch.ones(T, T, dtype=torch.bool, device=Q.device).tril()
    P = torch.softmax(S.masked_fill(~mask, float("-inf")), dim=1)
    return P @ V


def test_flash_attention():
    torch.manual_seed(0)
    T, D = 32, 16
    Q = torch.randn(T, D, device=DEVICE)
    K = torch.randn(T, D, device=DEVICE)
    V = torch.randn(T, D, device=DEVICE)
    ref = torch.softmax(Q @ K.T / math.sqrt(D), dim=1) @ V
    out = flash_attention_triton(Q, K, V)
    assert out.shape == (T, D)
    assert torch.allclose(out, ref, atol=1e-5, rtol=1e-5)

    ref_c = causal_reference(Q, K, V)
    out_c = flash_attention_triton(Q, K, V, causal=True)
    assert torch.allclose(out_c, ref_c, atol=1e-5, rtol=1e-5)


run_if_gpu(test_flash_attention)


In [ ]:
def test_flash_trace():
    '''The two-tile online-softmax trace for row 0 of the worked example.'''
    s = torch.tensor([1.41421356, 0.70710678])
    V = torch.tensor([[1.0, 2.0], [1.0, 0.0]])

    m = torch.tensor(float("-inf"))
    l = torch.tensor(0.0)
    acc = torch.zeros(2)
    trace = []

    for j in range(2):
        s_j = s[j:j + 1]
        m_new = torch.maximum(m, s_j.max())
        alpha = torch.exp(m - m_new)
        p = torch.exp(s_j - m_new)
        l = l * alpha + p.sum()
        acc = acc * alpha + p[0] * V[j]
        m = m_new
        trace.append((m.item(), l.item(), acc.clone()))

    assert abs(trace[0][0] - 1.4142) < 1e-4
    assert abs(trace[0][1] - 1.0000) < 1e-4
    assert torch.allclose(trace[0][2], torch.tensor([1.0, 2.0]), atol=1e-4)

    assert abs(trace[1][0] - 1.4142) < 1e-4
    assert abs(trace[1][1] - 1.4931) < 1e-4
    assert torch.allclose(trace[1][2], torch.tensor([1.4931, 2.0]), atol=1e-3)

    out = trace[-1][2] / trace[-1][1]
    assert torch.allclose(out, torch.tensor([1.0, 1.3395]), atol=1e-4)


test_flash_trace()
print("flash trace: the two-tile online softmax matches the direct row")


## Where fusion stops

- **Legal fusion.** A later stage consumes a value elementwise or carries a running reduction.
- **Barrier.** A later stage needs the whole row (softmax denominator), a transpose that changes the tile layout, or the full output of a GEMM before the next tile can start.

| Boundary | Fusable? | Reason |
|---|---|---|
| LN → QKV | yes | row statistics and GEMM share the same tile |
| scores → softmax | online | running max and denominator |
| softmax → output | yes | $P$ tile feeds `tl.dot(p, v)` directly |
| output → residual | yes | epilogue vector add |
| MLP layer 1 → GELU → layer 2 | yes | GELU is elementwise |
| attention → MLP | no | the next stage needs the completed residual row |
| QKV → scores | no | every score tile needs all $d$ columns of $Q$ and $K$ |

- **In this chapter.** The block test composes the PyTorch reference; the fused MLP kernel is the first production-shaped fusion.


In [ ]:
@triton.jit
def fused_mlp_kernel(x_ptr, w1_ptr, b1_ptr, w2_ptr, b2_ptr, y_ptr, M,
                     D: tl.constexpr, H: tl.constexpr, BM: tl.constexpr):
    rows = tl.program_id(0) * BM + tl.arange(0, BM)
    d = tl.arange(0, D)
    h = tl.arange(0, H)

    x = tl.load(x_ptr + rows[:, None] * D + d[None, :],
                mask=rows[:, None] < M, other=0.0)
    w1 = tl.load(w1_ptr + d[:, None] * H + h[None, :])
    b1 = tl.load(b1_ptr + h)
    pre = tl.dot(x, w1, input_precision="ieee") + b1[None, :]

    # Exact GELU: 0.5*x*(1+erf(x/sqrt(2))).
    act = 0.5 * pre * (1.0 + tl.erf(pre * 0.7071067811865476))

    w2 = tl.load(w2_ptr + h[:, None] * D + d[None, :])
    b2 = tl.load(b2_ptr + d)
    y = tl.dot(act, w2, input_precision="ieee") + b2[None, :]

    tl.store(y_ptr + rows[:, None] * D + d[None, :], y,
             mask=rows[:, None] < M)


def fused_mlp_triton(X, W1, b1, W2, b2, BM=16):
    X, W1, b1, W2, b2 = X.contiguous(), W1.contiguous(), b1.contiguous(), W2.contiguous(), b2.contiguous()
    M, D = X.shape
    H = W1.shape[1]
    Y = torch.empty(M, D, device=X.device, dtype=torch.float32)
    grid = (triton.cdiv(M, BM),)
    fused_mlp_kernel[grid](X, W1, b1, W2, b2, Y, M, D=D, H=H, BM=BM)
    return Y


def test_fused_mlp():
    torch.manual_seed(0)
    M, D, H = 16, 16, 32
    X = torch.randn(M, D, device=DEVICE)
    W1 = torch.randn(D, H, device=DEVICE) * 0.2
    b1 = torch.randn(H, device=DEVICE) * 0.1
    W2 = torch.randn(H, D, device=DEVICE) * 0.2
    b2 = torch.randn(D, device=DEVICE) * 0.1

    Y = fused_mlp_triton(X, W1, b1, W2, b2)
    ref = mlp_ref(X, W1, b1, W2, b2)
    assert Y.shape == (M, D)
    assert torch.allclose(Y, ref, atol=1e-5, rtol=1e-5)


run_if_gpu(test_fused_mlp)


In [ ]:
def test_fused_block():
    '''A full pre-norm block in PyTorch, plus the residual identity.'''
    torch.manual_seed(0)
    T, d = 8, 16
    X = torch.randn(T, d)

    Wq = torch.randn(d, d) * 0.1
    Wk = torch.randn(d, d) * 0.1
    Wv = torch.randn(d, d) * 0.1
    Wo = torch.randn(d, d) * 0.1
    W1 = torch.randn(d, 4 * d) * 0.1
    b1 = torch.zeros(4 * d)
    W2 = torch.randn(4 * d, d) * 0.1
    b2 = torch.zeros(d)
    g1, beta1 = torch.ones(d), torch.zeros(d)
    g2, beta2 = torch.ones(d), torch.zeros(d)

    Y = block_ref(X, Wq, Wk, Wv, Wo, W1, b1, W2, b2, g1, beta1, g2, beta2)
    assert Y.shape == (T, d)

    # Explicit two-residual composition, checked against block_ref.
    h = X + attention_ref(layernorm_ref(X, g1, beta1) @ Wq,
                          layernorm_ref(X, g1, beta1) @ Wk,
                          layernorm_ref(X, g1, beta1) @ Wv)[0] @ Wo
    y = h + mlp_ref(layernorm_ref(h, g2, beta2), W1, b1, W2, b2)
    assert torch.allclose(Y, y, atol=1e-6)

    # Zero branches leave the residual stream untouched.
    Y0 = block_ref(X, Wq, Wk, torch.zeros(d, d), Wo, torch.zeros(d, 4 * d),
                   torch.zeros(4 * d), torch.zeros(4 * d, d), torch.zeros(d),
                   g1, beta1, g2, beta2)
    assert torch.allclose(Y0, X, atol=1e-6)


test_fused_block()
print("fused block: the two residual adds and the zero-branch identity hold")


## KV cache: shapes, append, and traffic

- **Purpose.** Makes autoregressive decoding linear in the context instead of quadratic in the number of generated tokens.
- **Layout.** For each layer and each of $K$ and $V$: $K^{\text{cache}},V^{\text{cache}}\in\mathbb R^{B\times H\times T\times d}$.
- **Bytes.** With $L$ layers and element size $b$, the cache is $2\,L\,B\,H\,T\,d\,b$ bytes.
- **Per step.** The kernel writes $2\,L\,B\,H\,d\,b$ bytes and reads $2\,L\,B\,H\,T\,d\,b$, so read/write $=T$.
- **Concrete count.** For $B=1$, $H=32$, $L=32$, $T=4096$, $d=128$, FP16 ($b=2$):

$$
\begin{aligned}
\text{cache}&=2\cdot32\cdot1\cdot32\cdot4096\cdot128\cdot2
=2{,}147{,}483{,}648\ \text{bytes}=2\ \text{GiB},\\
\text{write}&=2\cdot32\cdot1\cdot32\cdot128\cdot2
=524{,}288\ \text{bytes}=0.5\ \text{MiB}.
\end{aligned}
$$

- **Ratio.** The read/write ratio is $4096$.
- **FLOPs.** QK$^\top$ and PV each cost about $2\,L\,B\,H\,T\,d$ per generated token, so $2^{31}$ FLOPs total.
- **Intensity.** $\mathrm{AI}=2^{31}/2^{31}\approx1$ FLOP/byte, far below machine balance; decoding waits on the cache, not the tensor cores.
- **Why it matters.** This is why paged caches, grouped-query attention, and cache quantization matter.
- **Kernel address.** For contiguous $[B,H,T,d]$, use `((b * H + h) * T + pos) * d + c`; a paged cache replaces this with a page-table lookup.


In [ ]:
@triton.jit
def kv_append_kernel(k_ptr, v_ptr, k_cache_ptr, v_cache_ptr, T, D, pos,
                     BLOCK_D: tl.constexpr):
    bh = tl.program_id(0)
    c = tl.arange(0, BLOCK_D)
    mask = c < D

    src = bh * D + c
    dst = (bh * T + pos) * D + c

    k = tl.load(k_ptr + src, mask=mask, other=0.0)
    v = tl.load(v_ptr + src, mask=mask, other=0.0)
    tl.store(k_cache_ptr + dst, k, mask=mask)
    tl.store(v_cache_ptr + dst, v, mask=mask)


def kv_append_triton(k_new, v_new, k_cache, v_cache, pos, BLOCK_D=None):
    B, H, D = k_new.shape
    T = k_cache.shape[2]
    BLOCK_D = BLOCK_D or triton.next_power_of_2(D)
    kv_append_kernel[(B * H,)](k_new.contiguous(), v_new.contiguous(),
                               k_cache, v_cache, T, D, pos, BLOCK_D=BLOCK_D)
    return k_cache, v_cache


def test_kv_append():
    torch.manual_seed(0)
    B, H, T, D = 2, 4, 4, 8
    k_cache = torch.zeros(B, H, T, D, device=DEVICE)
    v_cache = torch.zeros(B, H, T, D, device=DEVICE)
    k_new = torch.randn(B, H, D, device=DEVICE)
    v_new = torch.randn(B, H, D, device=DEVICE)

    kv_append_triton(k_new, v_new, k_cache, v_cache, pos=2)
    assert torch.allclose(k_cache[:, :, 2, :], k_new, atol=1e-6)
    assert torch.allclose(v_cache[:, :, 2, :], v_new, atol=1e-6)
    assert torch.equal(k_cache[:, :, 0, :], torch.zeros(B, H, D, device=DEVICE))
    assert k_cache.shape == (B, H, T, D)


run_if_gpu(test_kv_append)


In [ ]:
def test_kv_traffic():
    B, H, L, T, d, b = 1, 32, 32, 4096, 128, 2
    cache_bytes = 2 * L * B * H * T * d * b
    write_bytes = 2 * L * B * H * d * b
    read_bytes = 2 * L * B * H * T * d * b

    assert cache_bytes == 2_147_483_648
    assert cache_bytes == 2 * 1024 ** 3
    assert write_bytes == 524_288
    assert write_bytes == 0.5 * 1024 ** 2
    assert read_bytes // write_bytes == T == 4096

    # QK^T and PV each cost 2*L*B*H*T*d FLOPs per generated token.
    flops = 2 * (2 * L * B * H * T * d)
    assert flops == 2_147_483_648
    assert flops // cache_bytes == 1  # arithmetic intensity about 1 FLOP/byte


test_kv_traffic()
print("KV cache: 2 GiB cache, 0.5 MiB write, read/write ratio 4096, AI ~ 1")


## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| Missing $1/\sqrt d$ | logits grow with $d$; softmax saturates to one-hot | scale the scores before the softmax |
| Mask applied after softmax | remaining probabilities no longer sum to one | add $-\infty$ to masked logits, then softmax |
| Causal mask off by one | token $i$ sees key $i$ when it should not (or misses itself) | use $j\le i$ for self-inclusive causal attention |
| Mask value `0` instead of `-inf` | masked keys still receive probability | use `-inf` before the row max |
| RoPE sign flipped | relative offset $p-s$ becomes $s-p$; scores change | match the checkpoint's convention and pair layout |
| LayerNorm $\epsilon$ outside the root | $\sqrt{\sigma^2}+\epsilon$ instead of $\sqrt{\sigma^2+\epsilon}$ | keep $\epsilon$ inside `tl.rsqrt(var + eps)` |
| LayerNorm variance with $d-1$ | a scale error that grows with $d$ | divide by $d$ (biased variance) |
| KV-cache stride assumed contiguous | wrong rows for paged/transposed caches | pass the stride or the page table explicitly |
| FP16 softmax | `exp` saturates, rows become one-hot or NaN | upcast logits to FP32 before `exp` |
| Transposed $Q$ or $K$ | shape error, or silently wrong scores when $T=d$ | check `Q @ K.T` against the reference shape |

## Self-check

1. For $T=4$, $d=3$, write the shapes of $Q,K,V,S,P,O$ and the number of elements in $S$. Why is $S$ square?
2. Recompute the worked example with $W_v=I$. What is $O$? (Answer: $O_{0}=(1,0.6698)$, $O_{1}=(1,0.5)$.)
3. Why does the online-softmax update rescale $\mathrm{acc}$ and $\ell$ by the same factor? What happens if only one is rescaled?
4. A causal mask uses $j\le i$. What changes if you use $j\lt i$? Which row becomes all $-\infty$?
5. For $B=2$, $H=8$, $L=12$, $T=2048$, $d=64$, FP16, compute the KV-cache size and the per-step read/write ratio.

## Chapter summary

- **Block.** Shape-preserving stages around a residual stream: embedding, position, QKV, attention, normalization, MLP.
- **Shape rule.** $[T_q,d]\times[T_k,d]\to[T_q,T_k]\to[T_q,d]$; the only square object is the score matrix.
- **Softmax.** Stable subtraction of the row maximum; FlashAttention does it incrementally with $(m,\ell,\mathrm{acc})$.
- **Norms.** LayerNorm subtracts the mean; RMSNorm does not; both normalize per row and apply a per-channel affine.
- **KV cache.** Turns decoding from recomputation into a memory read; the read/write ratio is the context length, so long-context inference is bandwidth-bound.

**Where this leads.**
- **Exercises 1–4.** Extend the embedding, position, RoPE, and QKV stages.
- **Exercises 5–8.** Push softmax, FlashAttention, LayerNorm, and MLP fusion.
- **Exercises 9–10.** Add a paged KV cache and compose a full Triton block.
- **Next chapter.** Chapter 24 wraps kernels in `autograd.Function`, checks gradients, and profiles the result.

## Exercises

1. **Embedding.** Extend the gather kernel to a 3-D id tensor $(B,T)$ and check the output shape $(B,T,d)$.
2. **Position.** Implement both the interleaved and the half-split sinusoidal layouts. Show which one matches the reference and where the channels move.
3. **RoPE.** Add a `PAIR_LAYOUT` compile-time flag for adjacent vs half-split pairs and test both against a PyTorch reference.
4. **QKV.** Write a fused QKV GEMM with $N=3d$ and an epilogue that writes three separate tensors. Compare its traffic with three separate GEMMs.
5. **Softmax.** Replace the row softmax with a two-pass online version and check that it matches `torch.softmax` for a row with a large logit.
6. **FlashAttention.** Add a causal flag (done in the kernel) and test a sequence where $T$ is not a multiple of $B_N$. Which tile is fully masked?
7. **LayerNorm.** Implement Welford's one-pass variance and compare it with the two-pass version on a row with a large mean.
8. **MLP.** Fuse the bias adds and the GELU into the GEMM epilogues and compare the number of HBM round trips with the unfused version.
9. **KV cache.** Add a batch dimension and a page table: map logical position $t$ to a physical page and offset, then append through the table.
10. **Block.** Compose the Triton flash kernel and the fused MLP into one block and check it against `block_ref` on a random $8\times16$ input.


In [ ]:
CPU_TESTS = [
    test_worked_example,
    test_method01_attention,
    test_method01_norms_mlp_block,
    test_flash_trace,
    test_fused_block,
    test_kv_traffic,
]

GPU_TESTS = [
    test_embedding_kernel,
    test_norm_kernel,
    test_row_softmax,
    test_attention_scores,
    test_attention_output,
    test_flash_attention,
    test_fused_mlp,
    test_kv_append,
]

print("CPU tests")
for fn in CPU_TESTS:
    fn()

print()
print("GPU tests (skipped without CUDA/ROCm)")
for fn in GPU_TESTS:
    run_if_gpu(fn)

print()
print("done: CPU checks passed; GPU checks ran or printed SKIP")
